In [4]:
pip install geopandas rioxarray xarray

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ---------------------------------------- 1.4/1.4 MB 13.4 MB/s  0:00:00

  Attempting uninstall: xarray

    Found existing installation: xarray 2025.10.1

    Uninstalling xarray-2025.10.1:

      Successfully uninstalled xarray-2025.10.1

   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 0/2 [xarray]
   ---------------------------------------- 2/2 [rioxarray]

Note: you may need to restart the kernel to use updated p

In [1]:
import dask
print(dask.__version__)

2026.3.0


In [6]:
"""
fdcf_points.py
==============
Módulo de extração e filtragem de focos de calor (FDCF) a partir de arquivos
NetCDF4 do produto ABI-L2-FDCF do satélite GOES-16.

Fluxo principal:
    1. Abertura do raster FDCF via rioxarray na projeção geoestacionária original.
    2. Recorte espacial aproximado pelo bounding box do Pantanal (na projeção nativa),
       antes de reprojetar — evita reprojetar o disco completo desnecessariamente.
    3. Reprojeção para SIRGAS 2000 (EPSG:4674) — coordenadas em graus decimais.
    4. Filtragem por qualidade: Mask == 10 (foco confirmado) e DQF == 0 (dado válido).
    5. Exportação por arquivo em subpastas: csv/, shapefile/, metadados/.
    6. Resumo consolidado ao final do processamento em lote.

Variáveis do produto FDCF utilizadas:
    - Mask  : classificação do pixel (10 = foco de calor ativo confirmado)
    - DQF   : flag de qualidade (0 = dado válido)
    - Power : potência radiativa do fogo (MW)

Dependências:
    xarray, rioxarray, geopandas, shapely, numpy, pandas

Uso típico:
    processar_pasta_completa(
        pasta_entrada='Dados/ABI-L2-FDCF/netCDF',
        pasta_saida='Arquivos/FDCF_DATA',
    )
"""

import json
import re
import warnings
from pathlib import Path
from typing import Dict, List, Optional

import geopandas as gpd
import numpy as np
import pandas as pd
import rioxarray as rxr
import xarray as xr
from shapely.geometry import Point, box

# Suprime UserWarnings do rasterio conhecidos e não-acionáveis durante
# a reprojeção de produtos GOES-16 (ex.: avisos de CRS geoestacionário).
warnings.filterwarnings("ignore", category=UserWarning, module="rasterio")

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Limites geográficos da área de estudo (Pantanal)
PANTANAL_BOUNDS: Dict[str, float] = {
    "xmin": -60.0,   # Longitude oeste
    "xmax": -53.0,   # Longitude leste
    "ymin": -22.0,   # Latitude sul
    "ymax": -15.0,   # Latitude norte
}

# CRS de destino: SIRGAS 2000 geográfico (graus decimais)
CRS_DESTINO = "EPSG:4674"

# CRS geográfico de referência para o bounding box de recorte
CRS_GEO = "EPSG:4326"

# Variáveis de interesse no produto FDCF
VARIAVEIS_FDCF = ["Mask", "Power", "DQF"]

# Critérios de filtragem de qualidade
MASK_FOCO_CONFIRMADO = 10   # pixel classificado como foco de calor ativo
DQF_VALIDO           = 0    # flag de qualidade indicando dado confiável

# Subpastas de exportação (relativas a pasta_saida)
PASTA_CSV = "csv"
PASTA_SHAPEFILE = "shapefile"
PASTA_METADADOS = "metadados"

# Padrão de data nos nomes de arquivo GOES-16: _sYYYYDDDHHMMSS
_PADRAO_DATA = re.compile(r"_s(\d{4})(\d{3})(\d+)")


# ---------------------------------------------------------------------------
# Funções auxiliares (uso interno)
# ---------------------------------------------------------------------------

def _extrair_data_do_nome(nome_arquivo: str) -> Optional[str]:
    """
    Extrai o identificador de data/hora do nome de um arquivo GOES-16.

    O padrão esperado é `_sYYYYDDDHHMMSS`, onde:
        YYYY = ano, DDD = dia juliano, HHMMSS = hora/minuto/segundo UTC.

    Parâmetros:
        nome_arquivo (str): Nome do arquivo (sem caminho).

    Retorna:
        str | None: String no formato 'YYYYDDDHHMMSS', ou None se não encontrado.

    Exemplos:
        >>> _extrair_data_do_nome('OR_ABI-L2-FDCF-M6_G16_s20201521350164_e(...).nc')
        '20201521350164'
    """
    match = _PADRAO_DATA.search(nome_arquivo)
    if match:
        return f"{match.group(1)}{match.group(2)}{match.group(3)}"
    return None


def _preparar_pastas_exportacao(pasta_saida: Path) -> Dict[str, Path]:
    """
    Cria as subpastas de exportação e retorna seus caminhos.

    Estrutura em `pasta_saida`:
        csv/         — arquivos .csv
        shapefile/   — shapefiles de pontos (.shp e auxiliares)
        metadados/   — arquivos .json de metadados por arquivo processado
    """
    pastas = {
        "csv": pasta_saida / PASTA_CSV,
        "shapefile": pasta_saida / PASTA_SHAPEFILE,
        "metadados": pasta_saida / PASTA_METADADOS,
    }
    for caminho in pastas.values():
        caminho.mkdir(parents=True, exist_ok=True)
    return pastas


def _recortar_e_reprojetar(
    arquivo: Path,
    bounds: Dict[str, float],
) -> xr.Dataset:
    """
    Abre um arquivo FDCF, recorta pela área de interesse e reprojeta para SIRGAS 2000.

    A ordem das operações — recorte antes da reprojeção — é intencional:
    evita reprojetar o disco completo do GOES-16 (5424×5424 pixels) quando
    apenas a região do Pantanal é necessária.

    O recorte é feito por índices de pixel (argmin sobre as coordenadas
    projetadas) após reprojetar o bounding box geográfico para o CRS nativo
    do arquivo. Embora aproximado, é eficiente e suficiente para o recorte
    preliminar — o filtro espacial exato é aplicado após a reprojeção.

    Parâmetros:
        arquivo (Path):       Caminho do arquivo NetCDF4.
        bounds  (dict):       Limites geográficos de recorte (xmin, xmax, ymin, ymax).

    Retorna:
        xr.Dataset: Dataset recortado e reprojetado para `CRS_DESTINO`.
    """
    # Abre o arquivo como raster via rioxarray
    ds = rxr.open_rasterio(arquivo, band_as_variable=False)

    # Remove a dimensão 'band' desnecessária para produtos 2D
    ds = ds.squeeze("band", drop=True)

    if ds.rio.crs:
        # Reprojeta o bounding box geográfico para o CRS nativo do arquivo
        # (projeção geoestacionária do GOES-16)
        bbox = box(bounds["xmin"], bounds["ymin"], bounds["xmax"], bounds["ymax"])
        bbox_gdf  = gpd.GeoDataFrame({"geometry": [bbox]}, crs=CRS_GEO)
        bbox_proj = bbox_gdf.to_crs(ds.rio.crs)

        x_coords = ds.x.values
        y_coords = ds.y.values

        # Encontra os índices de pixel mais próximos aos limites reprojetados
        ixmin = np.abs(x_coords - bbox_proj.total_bounds[0]).argmin()
        ixmax = np.abs(x_coords - bbox_proj.total_bounds[2]).argmin()
        iymin = np.abs(y_coords - bbox_proj.total_bounds[1]).argmin()
        iymax = np.abs(y_coords - bbox_proj.total_bounds[3]).argmin()

        # Garante ordem correta dos índices (y pode ser decrescente)
        ixmin, ixmax = sorted([ixmin, ixmax])
        iymin, iymax = sorted([iymin, iymax])

        # Recorte por índices na projeção original
        ds = ds.isel(x=slice(ixmin, ixmax), y=slice(iymin, iymax))

    # Reprojeta para SIRGAS 2000 — após o recorte, o subconjunto é menor
    ds = ds.rio.reproject(dst_crs=CRS_DESTINO)

    return ds


def _filtrar_focos(ds: xr.Dataset) -> pd.DataFrame:
    """
    Aplica filtros de qualidade e remove pixels sem dado, retornando um DataFrame.

    Critérios aplicados vetorialmente:
        - Mask == 10 : foco de calor confirmado
        - DQF  == 0  : dado de boa qualidade

    Após filtrar, remove linhas e colunas onde todas as variáveis de interesse
    são NaN, evitando linhas esparsas sem informação no DataFrame resultante.

    Parâmetros:
        ds (xr.Dataset): Dataset reprojetado com variáveis Mask e DQF.

    Retorna:
        pd.DataFrame: Registros que satisfazem os critérios de filtragem,
                      sem linhas completamente vazias.
                      DataFrame vazio se Mask ou DQF não estiverem presentes.
    """
    if "Mask" not in ds or "DQF" not in ds:
        print("  ⚠️  Variáveis 'Mask' ou 'DQF' não encontradas no dataset.")
        return ds.to_dataframe().reset_index()

    # Máscara booleana vetorizada: foco confirmado + boa qualidade
    mascara_valida = (ds["Mask"].values == MASK_FOCO_CONFIRMADO) & \
                     (ds["DQF"].values  == DQF_VALIDO)

    # Aplica filtro — pixels inválidos tornam-se NaN
    ds_filtrado = ds.where(mascara_valida)

    # Variáveis a considerar para remoção de linhas/colunas vazias
    # (exclui coordenadas e variáveis auxiliares geradas pelo rioxarray)
    vars_interesse = [
        v for v in ds_filtrado.data_vars
        if v not in ("x", "y", "spatial_ref")
    ]

    # Remove fatias inteiramente NaN em x e y para cada variável de interesse
    for var in vars_interesse:
        ds_filtrado = ds_filtrado.dropna(dim="x", how="all", subset=[var])
        ds_filtrado = ds_filtrado.dropna(dim="y", how="all", subset=[var])

    df = ds_filtrado.to_dataframe().reset_index()

    # Remove linhas onde todas as variáveis de interesse são NaN
    if vars_interesse:
        df = df.dropna(subset=vars_interesse, how="all")

    return df


def _salvar_shapefile(
    df: pd.DataFrame,
    caminho: Path,
) -> Optional[gpd.GeoDataFrame]:
    """
    Salva um DataFrame de focos como shapefile de pontos.

    Parâmetros:
        df      (pd.DataFrame): DataFrame com colunas 'lon' e 'lat'.
        caminho (Path):         Caminho completo do arquivo de saída.

    Retorna:
        GeoDataFrame | None: GeoDataFrame salvo, ou None se o DataFrame estiver vazio.
    """
    if df.empty:
        print(f"  ⚠️  Sem dados para salvar como shapefile: {caminho.name}")
        return None

    caminho.parent.mkdir(parents=True, exist_ok=True)
    geometry = [Point(lon, lat) for lon, lat in zip(df["lon"], df["lat"])]
    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=CRS_DESTINO)
    gdf.to_file(caminho)
    print(f"  🗺️  Shapefile salvo: {caminho.name}")
    return gdf


# ---------------------------------------------------------------------------
# Funções públicas
# ---------------------------------------------------------------------------

def processar_arquivo(
    arquivo: Path,
    pasta_saida: Path,
    bounds: Dict[str, float] = PANTANAL_BOUNDS,
) -> bool:
    """
    Processa um único arquivo NetCDF4 do produto FDCF.

    Pipeline:
        1. Extração do identificador de data/hora do nome do arquivo.
        2. Recorte espacial e reprojeção para SIRGAS 2000.
        3. Filtragem por qualidade (Mask == 10, DQF == 0).
        4. Renomeação de coordenadas (x → lon, y → lat).
        5. Salvamento em subpastas: csv/, shapefile/, metadados/.

    Parâmetros:
        arquivo     (Path): Caminho do arquivo NetCDF4.
        pasta_saida (Path): Diretório de saída para os produtos gerados.
        bounds      (dict): Limites geográficos de recorte (padrão: Pantanal).

    Retorna:
        bool: True se processado com sucesso, False caso contrário.
    """
    try:
        print(f"  → Processando: {arquivo.name}")

        # 1. Identificador de data/hora
        data_id = _extrair_data_do_nome(arquivo.name)
        if data_id is None:
            data_id = arquivo.stem
            print(f"  ⚠️  Padrão de data não encontrado, usando: {data_id}")
        else:
            print(f"  📅 Data extraída: {data_id}")

        # 2. Recorte e reprojeção
        ds = _recortar_e_reprojetar(arquivo, bounds)

        # Seleciona apenas as variáveis de interesse disponíveis no arquivo
        variaveis = [v for v in VARIAVEIS_FDCF if v in ds.data_vars]
        ds = ds[variaveis]

        # 3. Filtragem por qualidade
        df = _filtrar_focos(ds)

        # Adiciona timestamp do arquivo como coluna, se disponível
        if "time_coverage_start" in ds.attrs:
            df["time"] = pd.to_datetime(ds.attrs["time_coverage_start"])

        # 4. Renomeia coordenadas para nomes semânticos
        df = df.rename(columns={"x": "lon", "y": "lat"})

        # data_id já identifica o instante; lat+lon distinguem focos simultâneos
        df["key"] = (
            data_id + "_" +
            df["lat"].round(6).astype(str) + "_" +
            df["lon"].round(6).astype(str)
        )

        # 5. Exportação (uma subpasta por tipo de produto)
        pastas = _preparar_pastas_exportacao(pasta_saida)
        csv_path = pastas["csv"] / f"dados_filtrados_{data_id}.csv"
        shp_path = pastas["shapefile"] / f"focos_{data_id}.shp"
        meta_path = pastas["metadados"] / f"metadata_{data_id}.json"

        df.to_csv(csv_path, index=False, encoding="utf-8")
        print(f"  💾 CSV salvo: {csv_path.relative_to(pasta_saida)}")

        _salvar_shapefile(df, shp_path)

        # Metadados por arquivo (mantidos para rastreabilidade individual)
        metadados = {
            "arquivo_original":    arquivo.name,
            "data_identificador":  data_id,
            "data_processamento":  pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
            "numero_registros":    len(df),
            "bounds_utilizados":   bounds,
            "variaveis_processadas": variaveis,
        }
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(metadados, f, indent=4, ensure_ascii=False, default=str)
        print(f"  📋 Metadados salvos: {meta_path.relative_to(pasta_saida)}")

        print(f"  ✅ Processado com sucesso! ({len(df)} registros)")
        return True

    except Exception as e:
        print(f"  ❌ Erro ao processar {arquivo.name}: {e}")
        return False


def processar_pasta_completa(
    pasta_entrada: str | Path,
    pasta_saida: Optional[str | Path] = None,
    bounds: Dict[str, float] = PANTANAL_BOUNDS,
    extensoes: Optional[List[str]] = None,
) -> None:
    """
    Processa todos os arquivos NetCDF4 de uma pasta e salva os resultados.

    Parâmetros:
        pasta_entrada (str | Path): Diretório contendo os arquivos `.nc`.
        pasta_saida   (str | Path): Diretório de saída (padrão: pasta_entrada/resultados).
        bounds        (dict):       Limites geográficos de recorte (padrão: Pantanal).
        extensoes     (list[str]):  Extensões aceitas (padrão: ['.nc']).

    Levanta:
        FileNotFoundError: Se `pasta_entrada` não existir.
    """
    pasta_entrada = Path(pasta_entrada)
    if not pasta_entrada.exists():
        raise FileNotFoundError(f"Pasta de entrada não encontrada: '{pasta_entrada}'")

    pasta_saida = Path(pasta_saida) if pasta_saida else pasta_entrada / "resultados"
    pasta_saida.mkdir(parents=True, exist_ok=True)
    pastas_export = _preparar_pastas_exportacao(pasta_saida)

    extensoes = extensoes or [".nc"]

    # Listagem ordenada para comportamento determinístico entre execuções
    arquivos = sorted(
        f for f in pasta_entrada.iterdir()
        if f.is_file() and f.suffix.lower() in extensoes
    )

    if not arquivos:
        print(f"⚠️  Nenhum arquivo com extensões {extensoes} encontrado em '{pasta_entrada}'")
        return

    print(f"📁 Entrada : {pasta_entrada}")
    print(f"📁 Saída   : {pasta_saida}")
    print(f"  ├── {PASTA_CSV}/")
    print(f"  ├── {PASTA_SHAPEFILE}/")
    print(f"  └── {PASTA_METADADOS}/")
    print(f"📊 Arquivos: {len(arquivos)}")
    print("=" * 60)

    sucesso = 0
    erros   = 0
    com_data = 0
    sem_data = 0

    for idx, arquivo in enumerate(arquivos, start=1):
        print(f"\n[{idx}/{len(arquivos)}] {arquivo.name}")
        print("-" * 40)

        # Conta arquivos com/sem padrão de data para o resumo final
        if _extrair_data_do_nome(arquivo.name):
            com_data += 1
        else:
            sem_data += 1

        if processar_arquivo(arquivo, pasta_saida, bounds):
            sucesso += 1
        else:
            erros += 1

    print("\n" + "=" * 60)
    print("📈 RESUMO DO PROCESSAMENTO:")
    print(f"  ✅ Sucesso        : {sucesso}/{len(arquivos)}")
    print(f"  ❌ Erros          : {erros}/{len(arquivos)}")
    print(f"  📅 Com data       : {com_data}")
    print(f"  ⚠️  Sem data       : {sem_data}")
    print(f"  💾 CSV            : {pastas_export['csv']}")
    print(f"  🗺️  Shapefile      : {pastas_export['shapefile']}")
    print(f"  📋 Metadados      : {pastas_export['metadados']}")
    print("=" * 60)


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    PASTA_ENTRADA = Path("Dados") / "ABI-L2-FDCF" / "netCDF"
    PASTA_SAIDA   = Path("Arquivos") / "FDCF_DATA"

    processar_pasta_completa(
        pasta_entrada=PASTA_ENTRADA,
        pasta_saida=PASTA_SAIDA,
    )


📁 Entrada : Dados\ABI-L2-FDCF\netCDF
📁 Saída   : Arquivos\FDCF_DATA
  ├── csv/
  ├── shapefile/
  └── metadados/
📊 Arquivos: 5274

[1/5274] OR_ABI-L2-FDCF-M4_G16_s20201701605224_e20201701610027_c20201701610209.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M4_G16_s20201701605224_e20201701610027_c20201701610209.nc
  📅 Data extraída: 20201701605224
  💾 CSV salvo: csv\dados_filtrados_20201701605224.csv
  🗺️  Shapefile salvo: focos_20201701605224.shp
  📋 Metadados salvos: metadados\metadata_20201701605224.json
  ✅ Processado com sucesso! (6 registros)

[2/5274] OR_ABI-L2-FDCF-M6_G16_s20201301300147_e20201301309455_c20201301309574.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20201301300147_e20201301309455_c20201301309574.nc
  📅 Data extraída: 20201301300147


C:\Users\Samuca\AppData\Local\Temp\ipykernel_10264\698624672.py:258: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(caminho)
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'goes_imager_projection' to 'goes_image'
  ogr_write(
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field time created as String field, though DateTime requested.
  ogr_write(


  💾 CSV salvo: csv\dados_filtrados_20201301300147.csv
  ⚠️  Sem dados para salvar como shapefile: focos_20201301300147.shp
  📋 Metadados salvos: metadados\metadata_20201301300147.json
  ✅ Processado com sucesso! (0 registros)

[3/5274] OR_ABI-L2-FDCF-M6_G16_s20201301310147_e20201301319455_c20201301319584.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20201301310147_e20201301319455_c20201301319584.nc
  📅 Data extraída: 20201301310147
  💾 CSV salvo: csv\dados_filtrados_20201301310147.csv
  🗺️  Shapefile salvo: focos_20201301310147.shp
  📋 Metadados salvos: metadados\metadata_20201301310147.json
  ✅ Processado com sucesso! (1 registros)

[4/5274] OR_ABI-L2-FDCF-M6_G16_s20201301320147_e20201301329455_c20201301329584.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20201301320147_e20201301329455_c20201301329584.nc
  📅 Data extraída: 20201301320147


C:\Users\Samuca\AppData\Local\Temp\ipykernel_10264\698624672.py:258: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(caminho)
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'goes_imager_projection' to 'goes_image'
  ogr_write(
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field time created as String field, though DateTime requested.
  ogr_write(


  💾 CSV salvo: csv\dados_filtrados_20201301320147.csv
  🗺️  Shapefile salvo: focos_20201301320147.shp
  📋 Metadados salvos: metadados\metadata_20201301320147.json
  ✅ Processado com sucesso! (1 registros)

[5/5274] OR_ABI-L2-FDCF-M6_G16_s20201301330147_e20201301339455_c20201301339595.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20201301330147_e20201301339455_c20201301339595.nc
  📅 Data extraída: 20201301330147


...

C:\Users\Samuca\AppData\Local\Temp\ipykernel_10264\698624672.py:258: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(caminho)
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'goes_imager_projection' to 'goes_image'
  ogr_write(
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field time created as String field, though DateTime requested.
  ogr_write(


  💾 CSV salvo: csv\dados_filtrados_20202402020202.csv
  ⚠️  Sem dados para salvar como shapefile: focos_20202402020202.shp
  📋 Metadados salvos: metadados\metadata_20202402020202.json
  ✅ Processado com sucesso! (0 registros)

[5272/5274] OR_ABI-L2-FDCF-M6_G16_s20202402030202_e20202402039510_c20202402040051.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20202402030202_e20202402039510_c20202402040051.nc
  📅 Data extraída: 20202402030202
  💾 CSV salvo: csv\dados_filtrados_20202402030202.csv
  ⚠️  Sem dados para salvar como shapefile: focos_20202402030202.shp
  📋 Metadados salvos: metadados\metadata_20202402030202.json
  ✅ Processado com sucesso! (0 registros)

[5273/5274] OR_ABI-L2-FDCF-M6_G16_s20202402040202_e20202402049510_c20202402050114.nc
----------------------------------------
  → Processando: OR_ABI-L2-FDCF-M6_G16_s20202402040202_e20202402049510_c20202402050114.nc
  📅 Data extraída: 20202402040202
  💾 CSV salvo: csv\dados_filtrados_20202402040

C:\Users\Samuca\AppData\Local\Temp\ipykernel_10264\698624672.py:258: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(caminho)
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'goes_imager_projection' to 'goes_image'
  ogr_write(
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field time created as String field, though DateTime requested.
  ogr_write(


  💾 CSV salvo: csv\dados_filtrados_20202402050202.csv
  🗺️  Shapefile salvo: focos_20202402050202.shp
  📋 Metadados salvos: metadados\metadata_20202402050202.json
  ✅ Processado com sucesso! (1 registros)

📈 RESUMO DO PROCESSAMENTO:
  ✅ Sucesso        : 5274/5274
  ❌ Erros          : 0/5274
  📅 Com data       : 5274
  ⚠️  Sem data       : 0
  💾 CSV            : Arquivos\FDCF_DATA\csv
  🗺️  Shapefile      : Arquivos\FDCF_DATA\shapefile
  📋 Metadados      : Arquivos\FDCF_DATA\metadados


C:\Users\Samuca\AppData\Local\Temp\ipykernel_10264\698624672.py:258: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(caminho)
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'goes_imager_projection' to 'goes_image'
  ogr_write(
C:\Users\Samuca\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field time created as String field, though DateTime requested.
  ogr_write(
